In [14]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import icarusplot

In [15]:
WD = '/Users/jskim/Documents/Neutrino/ICARUS/GUNDAM-1.9.X/work'

In [16]:
np.set_printoptions(precision=4, suppress=True)

In [17]:
# Number of total variable bin
NBins = 8

PriorError_LoW = 0.4
PriorError_HiW = 0.6

PriorLogNormalError_LoW = (abs(np.log(1+PriorError_LoW))+abs(np.log(1-PriorError_LoW)))/2.
PriorLogNormalError_HiW = (abs(np.log(1+PriorError_HiW))+abs(np.log(1-PriorError_HiW)))/2.

print(PriorError_LoW, PriorLogNormalError_LoW)
print(PriorError_HiW, PriorLogNormalError_HiW)


0.4 0.42364893019360184
0.6 0.6931471805599453


In [18]:
CovName = f'RegCov_LogNormal_PriorLoW{PriorError_LoW:.1f}_PriorHiW{PriorError_HiW:.1f}'
print(CovName)

RegCov_LogNormal_PriorLoW0.4_PriorHiW0.6


In [19]:
PriorError_LoW

0.4

In [20]:
import ROOT

NSet = 2

OutputFileName = f'{CovName}.root'
f_out = ROOT.TFile(f'{WD}/Configs_ParameterSet/BackgroundFit/{OutputFileName}', 'RECREATE')

h_Cov_TMat = ROOT.TMatrixDSym(NBins * NSet)

for i_Set in range(NSet):
    global_offset = i_Set * NBins
    for i in range(NBins):
        for j in range(NBins):
            if i+global_offset==j+global_offset:
                if i_Set==0:
                    h_Cov_TMat[i+global_offset][j+global_offset] = PriorLogNormalError_LoW*PriorLogNormalError_LoW
                else:
                    h_Cov_TMat[i+global_offset][j+global_offset] = PriorLogNormalError_HiW*PriorLogNormalError_HiW
            else:
                h_Cov_TMat[i+global_offset][j+global_offset] = 0

h_Cov_TMat.SetTol(1E-20)
h_Cov_TMat.Write(f'{CovName}')

param_prior = ROOT.TVectorD(NSet*NBins)
for i in range(NSet*NBins):
    param_prior[i] = 0.
param_prior.Write("PriorVals")

f_out.cd()
f_out.Close()